# Triton 编程范式 - 课后练习

本 Notebook 包含三个练习，帮助你巩固 Triton 的核心概念。

**学习目标**：
- 掌握 Triton 的基本语法和向量化操作
- 理解 `BLOCK_SIZE` 对性能的影响
- 学会用向量化方式处理复杂的数据访问模式

In [ ]:
import torch
import triton
import triton.language as tl
import matplotlib.pyplot as plt
import time

# 检查 GPU 可用性
assert torch.cuda.is_available(), "需要 CUDA 支持的 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Triton version: {triton.__version__}")

---

## 练习 1: AXPY 操作

**目标**：实现 BLAS 标准的 AXPY 操作：$Z = \alpha \cdot X + Y$

这是最基础的练习，帮助你熟悉 Triton 的基本模式。

**提示**：
- 结构与 `vector_add` 几乎相同
- `alpha` 是标量，可以直接与向量相乘（自动广播）
- 不需要对 `alpha` 使用 `tl.load`

In [ ]:
@triton.jit
def axpy_kernel(
    x_ptr, y_ptr, z_ptr,
    n_elements,
    alpha,  # 标量参数
    BLOCK_SIZE: tl.constexpr
):
    """
    TODO: 实现 AXPY 操作
    1. 计算 pid 和 offsets
    2. 创建 mask
    3. 加载 x 和 y
    4. 计算 z = alpha * x + y
    5. 存储 z
    """
    # ==================== 在下方编写代码 ====================
    
    
    
    # ========================================================
    pass

def run_axpy(x, y, alpha):
    n_elements = x.numel()
    z = torch.empty_like(x)
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    
    axpy_kernel[grid](
        x, y, z,
        n_elements, alpha,
        BLOCK_SIZE=1024
    )
    return z

In [ ]:
# 测试 AXPY
size = 98432
alpha = 3.14
x = torch.randn(size, device='cuda', dtype=torch.float32)
y = torch.randn(size, device='cuda', dtype=torch.float32)

# Triton 实现
z_triton = run_axpy(x, y, alpha)

# PyTorch 参考实现
z_torch = alpha * x + y

# 验证
if torch.allclose(z_triton, z_torch, atol=1e-5):
    print("AXPY 测试通过！")
else:
    print("AXPY 测试失败！")
    print(f"最大误差: {torch.max(torch.abs(z_triton - z_torch)).item():.2e}")

**思考题**：
1. 为什么 `alpha` 不需要 `tl.load`？
2. 如果 `alpha` 是一个向量（每个元素有不同的系数），代码需要怎么改？

---

## 练习 2: 性能测试 - BLOCK_SIZE 的影响

**目标**：探索不同 `BLOCK_SIZE` 对性能的影响，找出最优配置

这个练习帮助你理解为什么 Triton 的 `BLOCK_SIZE` 通常比 CUDA 的 `blockDim` 大得多。

**测试方案**：
- 使用向量加法作为基准测试
- 测试不同的 `BLOCK_SIZE`: [128, 256, 512, 1024, 2048, 4096]
- 测量执行时间和内存带宽

In [ ]:
# 向量加法 Kernel（用于性能测试）
@triton.jit
def add_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    tl.store(output_ptr + offsets, output, mask=mask)

In [ ]:
def benchmark_block_size(block_size, x, y, output, warmup=10, repeat=100):
    """基准测试单个 BLOCK_SIZE"""
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    
    # Warmup
    for _ in range(warmup):
        add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=block_size)
    
    # Timing
    torch.cuda.synchronize()
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    start_event.record()
    for _ in range(repeat):
        add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=block_size)
    end_event.record()
    
    torch.cuda.synchronize()
    time_ms = start_event.elapsed_time(end_event) / repeat
    
    # 计算带宽 (读 x, 读 y, 写 output)
    total_bytes = 3 * n_elements * 4  # float32 = 4 bytes
    bandwidth_gb_s = total_bytes / (time_ms * 1e-3) / 1e9
    
    return time_ms, bandwidth_gb_s

In [ ]:
# 运行基准测试
size = 1024 * 1024 * 10  # 10M elements
x = torch.randn(size, device='cuda', dtype=torch.float32)
y = torch.randn(size, device='cuda', dtype=torch.float32)
output = torch.empty_like(x)

block_sizes = [128, 256, 512, 1024, 2048, 4096]
results = []

print(f"{'BLOCK_SIZE':<15} {'Time (ms)':<15} {'Bandwidth (GB/s)':<20}")
print("-" * 50)

for bs in block_sizes:
    time_ms, bandwidth = benchmark_block_size(bs, x, y, output)
    results.append((bs, time_ms, bandwidth))
    print(f"{bs:<15} {time_ms:<15.3f} {bandwidth:<20.2f}")

In [ ]:
# 可视化结果
block_sizes_list = [r[0] for r in results]
bandwidths = [r[2] for r in results]

plt.figure(figsize=(10, 5))
plt.plot(block_sizes_list, bandwidths, marker='o', linewidth=2, markersize=8)
plt.xlabel('BLOCK_SIZE', fontsize=12)
plt.ylabel('Bandwidth (GB/s)', fontsize=12)
plt.title('Triton BLOCK_SIZE vs Memory Bandwidth', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xscale('log', base=2)
plt.xticks(block_sizes_list, block_sizes_list)

# 标注最佳 BLOCK_SIZE
best_idx = bandwidths.index(max(bandwidths))
plt.axvline(x=block_sizes_list[best_idx], color='r', linestyle='--', alpha=0.5)
plt.text(block_sizes_list[best_idx], max(bandwidths) * 0.95, 
         f'Best: {block_sizes_list[best_idx]}', ha='center', fontsize=10, color='r')

plt.tight_layout()
plt.show()

print(f"\n🏆 最优 BLOCK_SIZE: {block_sizes_list[best_idx]}")
print(f"🏆 最高带宽: {max(bandwidths):.2f} GB/s")

**思考题**：
1. 为什么 `BLOCK_SIZE=128` 性能较差？（提示：GPU 利用率）
2. 为什么 `BLOCK_SIZE=4096` 可能也不理想？（提示：寄存器压力）
3. 对比 CUDA 的 `blockDim.x` 常用值（256），Triton 的最优 `BLOCK_SIZE` 为什么更大？

---

## 练习 3: 1D 卷积（挑战）

**目标**：实现简单的 1D 卷积（3-tap box filter）：$Y[i] = X[i-1] + X[i] + X[i+1]$

边界条件：超出边界的值用 0 填充

**难点**：
- 需要访问相邻元素（左邻居和右邻居）
- 边界处理：`i=0` 时左邻居不存在，`i=n-1` 时右邻居不存在
- 需要为不同的加载操作创建不同的 mask

**提示**：分别加载三次
```

In [ ]:
@triton.jit
def conv1d_kernel(
    x_ptr, y_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr
):
    """
    TODO: 实现 3-tap 1D 卷积
    Y[i] = X[i-1] + X[i] + X[i+1]
    """
    # ==================== 在下方编写代码 ====================
    
    
    
    # ========================================================
    pass

def run_conv1d(x):
    n_elements = x.numel()
    y = torch.empty_like(x)
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    
    conv1d_kernel[grid](
        x, y,
        n_elements,
        BLOCK_SIZE=1024
    )
    return y

In [ ]:
# 测试 1D 卷积
size = 1024
x = torch.randn(size, device='cuda', dtype=torch.float32)

# Triton 实现
y_triton = run_conv1d(x)

# PyTorch 参考实现
x_padded = torch.nn.functional.pad(x, (1, 1), mode='constant', value=0)
y_torch = x_padded[:-2] + x_padded[1:-1] + x_padded[2:]

# 验证
if torch.allclose(y_triton, y_torch, atol=1e-5):
    print("Conv1D 测试通过！")
else:
    print("Conv1D 测试失败！")
    print(f"最大误差: {torch.max(torch.abs(y_triton - y_torch)).item():.2e}")
    
    # 显示前几个元素用于调试
    print("\n前 5 个元素对比:")
    print(f"Triton: {y_triton[:5].cpu().numpy()}")
    print(f"Torch:  {y_torch[:5].cpu().numpy()}")

In [ ]:
# 可视化卷积效果（可选）
size = 100
x = torch.randn(size, device='cuda', dtype=torch.float32)
y = run_conv1d(x)

plt.figure(figsize=(12, 5))
plt.plot(x.cpu().numpy(), label='Input', alpha=0.7)
plt.plot(y.cpu().numpy(), label='Output (Smoothed)', alpha=0.7, linewidth=2)
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('1D Convolution: Box Filter (3-tap)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**思考题**（高级）：
1. 为什么这种方法效率不高？（提示：重复加载）
2. 如何优化？（提示：Shared Memory 或加载更大的块然后切片）

---

## 总结

完成这三个练习后，你应该：
- 掌握了 Triton kernel 的基本写法
- 理解了 `BLOCK_SIZE` 对性能的重要影响
- 学会了如何处理复杂的内存访问模式

**下一步**：学习 Triton 的 Shared Memory 和 Block Reduction 操作！